# Classification and routing

Classify once, then send the work to the cheapest handler that can do it: plain code, a specialist, or a person.


In [ ]:
import sys
from datetime import date
from pathlib import Path
import json
import re
import statistics

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from langchain_typesafe import Choice, Noul, NoulCriteria, Score
from jev_examples.settings import ask, ask_many, draft, jev_model, openai_ready, show, typesafe_ready
from jev_examples.sample_data import (
    corpus_docs,
    customers,
    emails,
    load_json,
    lookup_order,
    open_incidents,
    order,
    products,
    read_text,
    ticket,
    tickets,
)

print("Jev model:", jev_model())
print("Jev key set:", typesafe_ready())
print("OpenAI key set:", openai_ready())


## 23. Intent routing

Order status needs no language model. A product question can use a specialist. Low confidence goes to a person.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "intent": Choice(
            instructions="The primary intent of this message",
            criteria={
                "order_status": "Asking where an existing order is",
                "product_question": "Asking about a product before buying",
                "return_exchange": "Wants to return or exchange something",
                "complaint": "Unhappy and wants a resolution",
            },
        ),
        "complexity": Score(
            instructions="How hard is this to resolve?",
            criteria=["Standard", "Needs judgment", "Edge case"],
        ),
    }
    for item in tickets():
        if item["id"] not in ("T-118", "T-270", "T-250", "T-260"):
            continue
        response = ask(item["body"], questions)
        show(response)
        intent = response.choices["intent"]
        if intent.confidence < 0.5:
            route = "human"
        elif intent.choice == "order_status":
            route = "lookup_order"
        elif response.scores["complexity"].score > 1.4:
            route = "human"
        else:
            route = intent.choice
        print(item["id"], "->", route)


**What you should see.** `T-118` should be a lookup. `T-270` should be a product question. `T-250` should be a return or exchange. `T-260` is the angry missing tent and may go to a person because it is hard.


## 24. Ask every question you might need, then score in code

Fan-out: severity only matters for a bug, but you can ask it anyway. One call is enough. The priority formula is yours.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    response = ask(
        ticket("T-104")["body"],
        {
            "category": Choice(
                instructions="Broad category",
                criteria={"bug": "The site or an order is broken", "billing": "A charge or a refund", "feature": "A new idea", "account": "Login or profile"},
            ),
            "bug_severity": Score(
                instructions="How severe is the reported issue?",
                criteria=["Cosmetic", "Degraded", "Blocking"],
            ),
            "refund_requested": Noul(instructions="Does the user explicitly ask for a refund or a credit?"),
            "frustration": Score(
                instructions="How frustrated is the user?",
                criteria=["Calm", "Frustrated but civil", "Very angry"],
            ),
        },
    )
    show(response)
    priority = (
        0.5 * response.scores["bug_severity"].score / 2
        + 0.3 * response.scores["frustration"].score / 2
        + 0.2 * response.nouls["refund_requested"].noul
    )
    print("priority:", round(priority, 2))
    if response.choices["category"].choice == "billing" and response.nouls["refund_requested"].noul > 0.6:
        route = "billing_refund_queue"
    else:
        route = "backlog"
    print("route:", route)


**What you should see.** Category should be billing, the refund Noul should be high, and the route should be the billing queue.


## 25. Walk a small category tree

Each step is a Choice whose options are the children of the current node. This tree is three departments, not a full catalog.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    tree = load_json("taxonomy.json")
    listing = "32oz plastic bottle with a flip straw lid. Fits most bike cages."
    top = ask(listing, {"child": Choice(instructions="Which department fits this product?", criteria={key: ", ".join(value) if isinstance(value, dict) else str(value) for key, value in tree.items()})})
    show(top)
    department = top.choices["child"].choice
    node = tree[department]
    if isinstance(node, dict):
        child = ask(listing, {"child": Choice(instructions="Which group fits this product?", criteria={key: str(value) for key, value in node.items()})})
        show(child)
        print("path:", department, ">", child.choices["child"].choice)
    else:
        print("path:", department)


**What you should see.** The bike-cage bottle should land under Sporting Goods, then Cycling, rather than Home drinkware.


## 26. Fall back to the parent category when unsure

If the fine label is low confidence, report the broader one instead of a wrong specific one.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    groups = {
        "outdoor_retail": "Sells tents, packs, and bottles to shoppers",
        "trucking": "Operates a freight fleet",
        "software": "Sells software",
    }
    parent = {"outdoor_retail": "retail", "trucking": "transport", "software": "technology"}
    questions = {"group": Choice(instructions="Which industry group best fits `filing`?", criteria=groups)}
    for filing in load_json("content.json")["filings"]:
        response = ask({"filing": filing}, questions)
        show(response)
        answer = response.choices["group"]
        label = answer.choice if answer.confidence >= 0.65 else parent[answer.choice]
        print("label:", label)


**What you should see.** The gear-shop filing should stay `outdoor_retail`. The mixed shop-and-trucks filing is the one that may fall back to a parent label.


## 27. Triage an inbox in one batch

`ask_many` classifies every email with the same questions. The result is an action, not a summary.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "action": Choice(
            instructions="What should happen with this email?",
            criteria={
                "reply_now": "Needs a reply today",
                "schedule": "Handle this week",
                "archive": "No reply needed",
                "unsure": "A person should look",
            },
        ),
        "asks_for_decision": Noul(instructions="Does the email ask the recipient to decide or approve something?"),
        "deadline_pressure": Score(
            instructions="How time-sensitive is this?",
            criteria=["None", "This week", "Today or already late"],
        ),
    }
    inbox = [row for row in emails() if row["folder"] == "inbox"]
    requests = [{"state": {"subject": row["subject"], "body": row["body"]}, "questions": questions} for row in inbox]
    for row, response in zip(inbox, ask_many(requests)):
        action = response.choices["action"]
        if action.choice == "unsure" or action.confidence < 0.55:
            route = "human"
        elif response.nouls["asks_for_decision"].noul > 0.6 and response.scores["deadline_pressure"].score > 1.4:
            route = "urgent_decision"
        else:
            route = action.choice
        print(row["id"], row["subject"], "->", route)


**What you should see.** The cycle-count approval and the camp decision should look urgent. The newsletter and the thank-you note should archive or schedule, not page anyone.
